<a href="https://colab.research.google.com/github/missstechie/Lightweight-Semantic-SDG-Classifier-MVP-/blob/main/Lightweight%20Semantic%20SDG%20Classifier%20(Low-Compute%20MVP).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Lightweight Semantic SDG Classifier (MVP)

This project demonstrates a low-compute approach to mapping project descriptions to UN Sustainable Development Goals (SDGs) using semantic similarity.

Instead of training a heavy ML model, we use sentence embeddings to match input text with SDG descriptions.

###  Key Features
- Uses transformer-based embeddings
- Works without labeled training data
- Fast and cost-efficient
- Returns Top-3 SDG predictions

###  Use Case
Useful for tools like SDG tagging systems, open-source project classification, and policy analysis.

---

In [61]:
!pip install -q sentence-transformers pandas scikit-learn

In [62]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

In [63]:
sdgs = {
    "SDG 1": "No poverty: ending poverty, financial inclusion, access to resources",
    "SDG 2": "Zero hunger: food security, agriculture, nutrition",
    "SDG 3": "Good health and well-being: healthcare, disease prevention, medical access",
    "SDG 4": "Quality education: schools, learning, digital education",
    "SDG 5": "Gender equality: women empowerment, equal rights",
    "SDG 6": "Clean water and sanitation: drinking water, hygiene",
    "SDG 7": "Clean energy: renewable energy, solar, electricity",
    "SDG 8": "Economic growth: jobs, employment, financial growth",
    "SDG 9": "Innovation and infrastructure: technology, industry",
    "SDG 10": "Reduced inequalities: social inclusion",
    "SDG 11": "Sustainable cities: urban development",
    "SDG 12": "Responsible consumption: sustainability",
    "SDG 13": "Climate action: environment, global warming",
    "SDG 14": "Life below water: oceans, marine life",
    "SDG 15": "Life on land: forests, biodiversity",
    "SDG 16": "Peace and justice: governance, institutions",
    "SDG 17": "Partnerships: collaboration, global cooperation"
}

In [64]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [65]:
sdg_texts = list(sdgs.values())
sdg_embeddings = model.encode(sdg_texts)

def _predict_sdg_with_details(text):
    text_embedding = model.encode(text)
    similarities = cosine_similarity([text_embedding], sdg_embeddings)[0]

    sdg_preds = []
    for i, sim in enumerate(similarities):
        sdg_code = list(sdgs.keys())[i]
        sdg_description = sdgs[sdg_code]
        sdg_preds.append((sdg_code, sdg_description, sim))

    # Sort by similarity in descending order and return top 3
    return sorted(sdg_preds, key=lambda x: x[2], reverse=True)[:3]

def predict_sdg(text):
    preds_with_details = _predict_sdg_with_details(text)
    if preds_with_details:
        top_pred = preds_with_details[0]
        return (top_pred[0], top_pred[1]) # Return (sdg_code, sdg_description)
    return (None, None)

def predict_sdg_top3(text):
    preds_with_details = _predict_sdg_with_details(text)
    return [p[0] for p in preds_with_details] # Return list of sdg_codes

def evaluate(df_to_evaluate):
    correct = 0
    total_predictions = len(df_to_evaluate)

    print(f"Evaluating {total_predictions} projects...")
    for _, row in df_to_evaluate.iterrows():
        pred_code, _ = predict_sdg(row["project"])

        # print("Project:", row["project"])
        # print("Predicted:", pred_code)
        # print("Actual:", row["actual_sdg"])
        # print("-----")

        if pred_code == row["actual_sdg"]:
            correct += 1

    accuracy = correct / total_predictions
    print(f"Accuracy: {accuracy:.4f} ({correct}/{total_predictions})\n")

In [66]:
print(predict_sdg_top3("A mobile app that improves rural healthcare and education access"))


['SDG 1', 'SDG 3', 'SDG 4']


In [67]:
test_cases = [
    "A mobile app for maternal healthcare in rural areas",
    "AI platform for climate change monitoring",
    "Online education system for underprivileged students",
    "Solar panels for villages without electricity",
    "Microfinance platform for small businesses"
]

for t in test_cases:
    print("Input:", t)
    print("Top 3 SDGs:", predict_sdg_top3(t))
    print()

Input: A mobile app for maternal healthcare in rural areas
Top 3 SDGs: ['SDG 1', 'SDG 3', 'SDG 2']

Input: AI platform for climate change monitoring
Top 3 SDGs: ['SDG 13', 'SDG 7', 'SDG 17']

Input: Online education system for underprivileged students
Top 3 SDGs: ['SDG 4', 'SDG 1', 'SDG 10']

Input: Solar panels for villages without electricity
Top 3 SDGs: ['SDG 7', 'SDG 11', 'SDG 1']

Input: Microfinance platform for small businesses
Top 3 SDGs: ['SDG 1', 'SDG 9', 'SDG 8']



In [68]:
data = {
    "project": [
        "Mobile health app for villages",
        "Online education platform for students",
        "Clean drinking water system",
        "Solar energy for rural homes"
    ],
    "actual_sdg": [
        "SDG 3",
        "SDG 4",
        "SDG 6",
        "SDG 7"
    ]
}

df = pd.DataFrame(data)
df

,project,actual_sdg
0,Mobile health app for villages,SDG 3
1,Online education platform for students,SDG 4
2,Clean drinking water system,SDG 6
3,Solar energy for rural homes,SDG 7


In [69]:
import numpy as np

# More comprehensive DPG-based dataset (simulated)
dpg_data = {
    "project": [
        "Developing open-source software for sustainable agriculture",
        "Providing digital literacy training in underserved communities",
        "Implementing blockchain solutions for transparent supply chains",
        "Creating AI tools for early disease detection in rural areas",
        "Open data platform for climate change research",
        "Digital identity system for refugees",
        "Renewable energy microgrid for remote villages",
        "Affordable assistive technology for people with disabilities",
        "Online platform connecting small farmers to markets",
        "Satellite imagery analysis for deforestation monitoring"
    ],
    "actual_sdg": [
        "SDG 2", # Zero hunger
        "SDG 4", # Quality education
        "SDG 9", # Industry, Innovation, and Infrastructure (proxy for transparent supply chains)
        "SDG 3", # Good health and well-being
        "SDG 13", # Climate Action (proxy for climate change research)
        "SDG 16", # Peace, Justice, and Strong Institutions (proxy for digital identity)
        "SDG 7", # Affordable and clean energy
        "SDG 10", # Reduced inequalities (proxy for assistive technology)
        "SDG 2", # Zero hunger
        "SDG 15"  # Life on Land
    ]
}

# Expand SDG descriptions for the simulated dataset if new SDGs are introduced
# (Assuming SDG 9, 10, 13, 15, 16 might be new for this expanded dataset)
sdgs.update({
    "SDG 8": "Decent work and economic growth",
    "SDG 9": "Industry, innovation and infrastructure",
    "SDG 10": "Reduced inequalities",
    "SDG 11": "Sustainable cities and communities",
    "SDG 12": "Responsible consumption and production",
    "SDG 13": "Climate action",
    "SDG 14": "Life below water",
    "SDG 15": "Life on land",
    "SDG 16": "Peace, justice and strong institutions",
    "SDG 17": "Partnerships for the goals"
})

# Re-encode sdg_embeddings with the expanded list of SDGs
sdg_texts = list(sdgs.values())
sdg_embeddings = model.encode(sdg_texts)

df_dpg = pd.DataFrame(dpg_data)
df_dpg

,project,actual_sdg
0,Developing open-source software for sustainabl...,SDG 2
1,Providing digital literacy training in underse...,SDG 4
2,Implementing blockchain solutions for transpar...,SDG 9
3,Creating AI tools for early disease detection ...,SDG 3
4,Open data platform for climate change research,SDG 13
5,Digital identity system for refugees,SDG 16
6,Renewable energy microgrid for remote villages,SDG 7
7,Affordable assistive technology for people wit...,SDG 10
8,Online platform connecting small farmers to ma...,SDG 2
9,Satellite imagery analysis for deforestation m...,SDG 15


In [70]:
# Evaluate with the new DPG dataset
print("\n--- Evaluation on DPG Dataset ---")
evaluate(df_dpg)


--- Evaluation on DPG Dataset ---
Evaluating 10 projects...
Accuracy: 0.7000 (7/10)



In [71]:
from ipywidgets import Textarea, Button, VBox, Output, Layout
from IPython.display import display

print("### SDG Prediction Interface ###\n")

project_input = Textarea(
    value='',
    placeholder='Enter project description here...',
    description='Project:',
    disabled=False,
    layout=Layout(width='auto', height='100px')
)

predict_button = Button(description="Predict SDG")
output_area = Output()

def on_button_click(b):
    with output_area:
        output_area.clear_output()
        project_description = project_input.value
        if project_description:
            print(f"Analyzing project: '{project_description}'")
            predictions = _predict_sdg_with_details(project_description)
            print("\nPredicted Top 3 SDGs:")
            for sdg_code, sdg_desc, score in predictions:
                print(f"  - {sdg_code}: {sdg_desc} (Similarity: {score:.4f})")
        else:
            print("Please enter a project description.")

predict_button.on_click(on_button_click)

display(VBox([project_input, predict_button, output_area]))

### SDG Prediction Interface ###



##  Limitations
- Uses static SDG descriptions (can be expanded)
- Does not support multi-label classification fully
- No domain-specific fine-tuning

##  Future Improvements
- Integrate external APIs
- Train on real SDG-labeled datasets
- Deploy as a web service